In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

print(f"REPO_ROOT: {REPO_ROOT}")

REPO_ROOT: /Users/ryant/Github/ryantjx/cuthberto-carlos


In [2]:
from predict_factsmc import build_factsmc_model
from scripts.smc.process_data import process_data_pl
import jax
from tqdm import tqdm
import cuthbert

# Offline Filtering

In [3]:
import polars as pl

N = 100

pl_data, jax_data, id_to_name = process_data_pl(
    start_date="2020-01-01",
    end_date="2026-06-11",
    future_matches=False,
    max_goals=8,
)
pl_data.filter(
    (pl.col("home_team") == "England") | (pl.col("away_team") == "England")
).head(5)

date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,friendly,timestamp_days,home_timestamp_previous,away_timestamp_previous,home_team_id,away_team_id
date,str,str,i64,i64,str,str,str,bool,bool,i64,i64,i64,i64,i64
2020-09-05,"""Iceland""","""England""",0,1,"""UEFA Nations League""","""Reykjavík""","""Iceland""",false,false,242,12,0,104,68
2020-09-08,"""Denmark""","""England""",0,0,"""UEFA Nations League""","""Copenhagen""","""Denmark""",false,false,245,242,242,59,68
2020-10-08,"""England""","""Wales""",3,0,"""Friendly""","""London""","""England""",false,true,275,245,243,68,252
2020-10-11,"""England""","""Belgium""",2,1,"""UEFA Nations League""","""London""","""England""",false,false,278,275,275,68,22
2020-10-14,"""England""","""Denmark""",0,1,"""UEFA Nations League""","""London""","""England""",false,false,281,278,278,68,59


In [4]:
num_teams = len(id_to_name)
smc_filter, factorializer = build_factsmc_model(N=N, num_teams=num_teams)

keys = jax.random.split(jax.random.PRNGKey(0), num_teams)

In [8]:
# init_state: (F, N, 2)
# local_states: (T, F, N, 2)
# final_state: (F, N, 2)

init_state, local_states, final_state = cuthbert.factorial.filter(
    filter_obj=smc_filter,
    factorializer=factorializer,
    model_inputs=jax_data,
    output_factorial=False,
    key=jax.random.PRNGKey(0)
)

In [22]:
local_states.particles.shape

(6002, 2, 100, 2)

In [23]:
local_states.log_weights.shape

(6002, 2, 100)

In [30]:
import numpy as np

# local_states.particles: (T, 2, N, 2) — timesteps × (home, away) × particles × (attack, defence)
# local_states.log_weights: (T, 2, N)
# Timestep t in local_states corresponds to match index t+1 in jax_data (index 0 = init)

particles = np.array(local_states.particles)       # (T, 2, N, 2)
log_weights = np.array(local_states.log_weights)   # (T, 2, N)
weights = np.exp(log_weights - log_weights.max(axis=-1, keepdims=True))
weights = weights / weights.sum(axis=-1, keepdims=True)  # normalise per (T, F)

T, F, N, D = particles.shape

# Build index arrays
t_idx = np.arange(T) + 1  # +1 because local_states[0] = match index 1
home_ids = np.array(jax_data.home_team_id)[t_idx]
away_ids = np.array(jax_data.away_team_id)[t_idx]
timestamps = np.array(jax_data.timestamp)[t_idx]

# Factor 0 = home, factor 1 = away
team_ids = np.stack([home_ids, away_ids], axis=1)  # (T, 2)

# Broadcast to (T, 2, N)
ts_b = np.broadcast_to(timestamps[:, None, None], (T, F, N))
team_b = np.broadcast_to(team_ids[:, :, None], (T, F, N))
attack_b = particles[..., 0]
defence_b = particles[..., 1]
weight_b = weights

# Map team_id → country name via numpy lookup
id_name = {int(k): v for k, v in id_to_name.items()}
max_id = max(id_name.keys()) + 1
name_lut = np.array([id_name.get(i, f"Team {i}") for i in range(max_id)], dtype=object)
country_b = name_lut[team_b]  # (T, 2, N)

origin_date = pl_data["date"].min()

# Flatten to rows
df_local = pl.DataFrame({
    "timestamp": ts_b.ravel(),
    "country": country_b.ravel(),
    "attack": attack_b.ravel(),
    "defence": defence_b.ravel(),
    "weight": weight_b.ravel(),
}).with_columns(
    (pl.lit(origin_date) + pl.col("timestamp").cast(pl.Int64).cast(pl.Duration("day"))).alias("date")
)


df_local

ValueError: invalid `time_unit`

Expected one of {'ns','us','ms'}, got 'day'.

In [7]:
# # slice the first element to get the prior sample
# init_model_inputs = jax.tree.map(lambda x: x[0], jax_data)
# # prepare the initial state for the SMC filter
# factorial_state = smc_filter.init_prepare(init_model_inputs, key=keys[0])
# # factorialize the initial state for the SMC filter
# factorial_state = factorializer.factorialize_init_state(factorial_state, init_model_inputs)

# for t in tqdm(range(1, len(jax_data.match_index)), desc="Filtering"):
#     # get inputs for the current time step. i.e. ResultData for the t-th match
#     model_inputs_t = jax.tree.map(lambda x: x[t], jax_data)
#     # pulls the state of the two teams particles involved at time t
#     local_state = factorializer.extract_and_join(factorial_state, model_inputs_t)
#     # propagates particles forward through `propagate_sample`
#     prep_state = smc_filter.filter_prepare(model_inputs_t, key=keys[t])
#     # reweights particles based on `log_potential` function, a bivariate poisson in this case.
#     filtered_joint = smc_filter.filter_combine(local_state, prep_state)
#     # resamples and inserts update particles back into factorial state
#     factorial_state = factorializer.marginalize_and_insert(
#         filtered_joint, factorial_state, model_inputs_t
#     )

# Prediction